# 03 — NepBERTa Fine-tuning (Google Colab)

## ⚠️ Run this on Google Colab — not locally

Fine-tuning NepBERTa needs a GPU. Colab's free T4 is enough if you pick
`Runtime → Change runtime type → T4 GPU`. Expected wall time: ~15–30
minutes for 5 epochs on 28k rows with `max_length=64`.

## The workflow this notebook automates

```
(A) Clone repo + install transformers         ← one-off per Colab session
(B) Upload preprocessed train.csv / test.csv  ← pops a file picker
(C) Fine-tune NepBERTa via src.models.nepberta ← GPU heavy lifting
(D) Evaluate on test set (SAME code as Phase 2)
(E) Save model + append one row to all_models_results.csv
(F) Zip everything and download
```

Bring the downloaded `.zip` back to your local machine, unzip at the repo
root, and Phase 4 (task #11) will have what it needs to plot the LR vs
NepBERTa comparison.

### Prereq: you've already cloned the repo into Colab

If not:
```bash
!git clone <YOUR_REPO_URL> repo && cd repo
```
Then restart this notebook FROM INSIDE `repo/notebooks/`.

## Step A — Install dependencies

Colab already has `torch`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`,
`seaborn`, and `psutil`. We only need to add `transformers` (+ `accelerate`
for fp16 / distributed training). Pinned versions match `requirements-gpu.txt`
in the repo.

In [ ]:
# -q = quiet. Install takes ~20 seconds first time; cached on re-runs.
!pip install -q transformers accelerate

## Step B — Upload preprocessed data

`data/processed/` is gitignored (the CSVs are regenerable artifacts, not
source code), so `git clone` didn't bring them. Run the cell below, then
select BOTH `train.csv` and `test.csv` from your local machine in the
file picker.

In [ ]:
import os
from google.colab import files

# Ensure the destination folder exists (matches config.DATA_PROCESSED).
os.makedirs('data/processed', exist_ok=True)

# `files.upload()` returns {filename: bytes}. We don't use the bytes —
# we just need the files to land on Colab's disk, then move them.
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f'data/processed/{filename}')

print('Processed files in place:', os.listdir('data/processed'))

## Step 1 — Imports

Same pattern as Phase 2's notebook. `src.models.nepberta` imports succeed here
because Colab has `torch` + `transformers` (unlike the local environment).

In [ ]:
import sys
from pathlib import Path

# We're running from notebooks/; project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from src import config
from src import evaluation as ev
from src import visualizations as viz
from src.models import nepberta as nb

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 — Load the preprocessed data

Same CSVs, same split, SAME rows as Phase 2. This is what makes the
LR vs NepBERTa comparison honest.

In [ ]:
train = pd.read_csv(f'{config.DATA_PROCESSED}/train.csv')
test  = pd.read_csv(f'{config.DATA_PROCESSED}/test.csv')

print(f'Train: {train.shape}   Test: {test.shape}')
print('Train label balance:')
print((train['label'].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

## Step 3 — Fine-tune NepBERTa

This is the slow step (15–30 min on T4). We use the test set as validation
because this is a single-shot project — no separate dev split, no cross-run
hyperparameter tuning. The Trainer still uses it correctly:

- Computes metrics each epoch
- Keeps the checkpoint with the best `macro_f1`
- Stops early if no improvement for `patience=2` epochs

⚠️  If you get "CUDA out of memory" here, reduce `NEPBERTA_BATCH_SIZE` in
`src/config.py` from 16 → 8 → 4 and re-run.

In [ ]:
# Everything happens inside a Profiler so we capture training time + peak mem.
with ev.Profiler() as train_prof:
    trainer, model, tokenizer = nb.train_model(
        train_texts  = train['text'],
        train_labels = train['label'],
        val_texts    = test['text'],
        val_labels   = test['label'],
    )

print(f'\n=== Training finished ===')
print(f'Wall-clock:    {train_prof.elapsed_seconds/60:.1f} min')
print(f'Peak memory:   {train_prof.peak_memory_mb:.0f} MB')

## Step 4 — Evaluate on the test set

Reuses `src.evaluation.compute_performance_metrics` — the same function LR
used in Phase 2. That's how the two models get graded on identical rubrics.

In [ ]:
# Predict + measure inference cost.
with ev.Profiler() as infer_prof:
    y_pred = nb.predict(model, tokenizer, test['text'])

n_test = len(y_pred)
ms_per_sample = infer_prof.elapsed_seconds * 1000 / n_test
print(f'Inference on {n_test:,} rows: '
      f'{infer_prof.elapsed_seconds:.2f}s total, {ms_per_sample:.2f} ms/sample')
print(f'Peak memory during inference: {infer_prof.peak_memory_mb:.0f} MB')

In [ ]:
metrics = ev.compute_performance_metrics(test['label'], y_pred)
ev.print_metrics(metrics, title='NepBERTa — test set')

In [ ]:
# Raw-counts confusion matrix.
viz.plot_confusion_matrix(metrics['confusion_matrix'],
                          save_path='confusion_matrix_nepberta.png')

In [ ]:
# Row-normalised confusion matrix (per-class recall view).
viz.plot_confusion_matrix(metrics['confusion_matrix'],
                          normalize=True,
                          save_path='confusion_matrix_nepberta_normalized.png')

## Step 5 — Training-loss curve

The Trainer logs loss every 50 steps AND evaluates at the end of every
epoch. We aggregate to per-epoch averages so the curve is readable.

In [ ]:
from collections import defaultdict

log = trainer.state.log_history

# Average train loss per epoch.
train_by_epoch = defaultdict(list)
for entry in log:
    if 'loss' in entry and 'epoch' in entry:
        epoch_bucket = int(entry['epoch']) + 1   # 1-indexed for display
        train_by_epoch[epoch_bucket].append(entry['loss'])
train_losses = [sum(train_by_epoch[e]) / len(train_by_epoch[e])
                for e in sorted(train_by_epoch.keys())]

# Validation loss per epoch — logged once per epoch by the Trainer.
val_losses = [entry['eval_loss'] for entry in log if 'eval_loss' in entry]

# `plot_training_loss` expects equal-length lists; if early stopping ran,
# trim train to match val.
n = min(len(train_losses), len(val_losses))
viz.plot_training_loss(train_losses[:n], val_losses[:n],
                       save_path='training_loss_nepberta.png')

## Step 6 — Save model + append results row

Two artifacts:

1. `outputs/models/nepberta_finetuned/` — the trained checkpoint as a folder
   (`config.json`, `model.safetensors`, tokenizer vocab, etc.).
2. One new row in `outputs/results/all_models_results.csv` — same schema as
   LR's row, so Phase 4 can read both rows as a table and plot them.

In [ ]:
# Save the fine-tuned model + tokenizer.
model_path = nb.save_model(model, tokenizer)
size_mb = ev.model_size_mb(model_path)
print(f'Model directory size: {size_mb:.1f} MB')

In [ ]:
# Build one flat result row (same schema as LR's row in Phase 2).
result_row = {
    'model': 'nepberta',
    **ev.flatten_metrics(metrics),
    'training_time_s':         train_prof.elapsed_seconds,
    'inference_time_s':        infer_prof.elapsed_seconds,
    'inference_ms_per_sample': infer_prof.elapsed_seconds * 1000 / n_test,
    'peak_memory_mb':          infer_prof.peak_memory_mb,
    'model_size_mb':           ev.model_size_mb(model_path),
}
ev.save_results_row(result_row, 'all_models_results.csv')

# Echo the row back.
pd.DataFrame([result_row]).T.rename(columns={0: 'value'})

## Step 7 — Zip and download

Zipping `outputs/` into one file makes the "download and move back to
local repo" step a single command. Expect ~450 MB for the model itself
plus a few MB of figures + the CSV.

In [ ]:
!zip -rq nepberta_artifacts.zip outputs/
size = os.path.getsize('nepberta_artifacts.zip') / (1024**2)
print(f'nepberta_artifacts.zip is {size:.0f} MB')

from google.colab import files
files.download('nepberta_artifacts.zip')

## Back on your local machine

Unzip at the repo root:

```bash
cd /path/to/NepaliLanguageSentimentAnalysis
unzip ~/Downloads/nepberta_artifacts.zip
```

That drops:
- `outputs/models/nepberta_finetuned/` — 450 MB of model weights
- `outputs/figures/confusion_matrix_nepberta*.png` + `training_loss_nepberta.png`
- `outputs/results/all_models_results.csv` — now with TWO rows (LR + NepBERTa)

From there, Phase 4 (`04_comparison.ipynb`) reads both rows and produces the
side-by-side comparison figures for the paper.